# EMT three-phase constant-PQ load

This self-contained notebook builds and simulates a minimal three-phase EMT network directly with the DPsim Python API. It demonstrates that `PQLoad` keeps its requested three-phase active and reactive powers while the terminal voltage and the power set points change.

Network: `three-phase voltage source -> load bus -> PQLoad`

Scenario:

| Time | Change | Purpose |
|---:|---|---|
| 0.10 s | voltage 400 V -> 320 V | current must rise while P and Q stay constant |
| 0.20 s | voltage 320 V -> 440 V | current must fall while P and Q stay constant |
| 0.30 s | P 30 kW -> 60 kW | active-power set-point tracking |
| 0.40 s | Q 15 kvar -> 30 kvar | reactive-power set-point tracking |

Requirement: DPsim must be built from this source tree with the Python bindings enabled and installed in the active notebook environment.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import dpsimpy

from dpsimpy import Domain, Logger, Math, PhaseType, Simulation, Solver, SystemTopology
from dpsimpy import emt
from dpsimpy.emt import ph3

print("Python:", sys.version.split()[0], sys.executable)
print("dpsimpy:", dpsimpy.__file__)
print("NumPy:", np.__version__, np.__file__)

if not hasattr(ph3, "PQLoad"):
    raise RuntimeError(
        "PQLoad is missing from dpsimpy. Run 'python -m pip install -e . -v' "
        "in the repository root, restart Jupyter, and select that environment's kernel."
    )

## Parameters and set-point schedule

`P` and `Q` are total three-phase powers. `V_nom` and the voltage schedule are line-to-line RMS values. Positive P and Q mean active and inductive-reactive consumption.

In [ ]:
SIM_NAME = "EMT_Ph3_PQLoad_Python"
FREQUENCY = 50.0
TIME_STEP = 50e-6
FINAL_TIME = 0.5

V_NOMINAL = 400.0
P_INITIAL = 30e3
Q_INITIAL = 15e3
MINIMUM_VOLTAGE_PU = 0.1

EVENTS = pd.DataFrame(
    [
        (0.10, "V_ll_rms", 320.0, "V"),
        (0.20, "V_ll_rms", 440.0, "V"),
        (0.30, "P", 60e3, "W"),
        (0.40, "Q", 30e3, "var"),
    ],
    columns=["time_s", "attribute", "new_value", "unit"],
)
EVENTS

## Build the network in Python

The source and the new `PQLoad` are instantiated, parameterized, connected, and placed in a `SystemTopology` here. No external C++ executable or pre-generated CSV is required.

In [ ]:
def three_phase_voltage(line_to_line_rms):
    return Math.single_phase_variable_to_three_phase(complex(line_to_line_rms, 0.0))


voltage_nominal = three_phase_voltage(V_NOMINAL)

load_bus = emt.SimNode("load_bus", PhaseType.ABC)
load_bus.set_initial_voltage(voltage_nominal)

source = ph3.VoltageSource("source")
source.set_parameters(voltage_nominal, FREQUENCY)

load = ph3.PQLoad("load")
load.set_parameters(
    active_power=P_INITIAL,
    reactive_power=Q_INITIAL,
    nominal_voltage=V_NOMINAL,
    minimum_voltage_per_unit=MINIMUM_VOLTAGE_PU,
)
load.set_max_iterations(20)
load.set_tolerance(1e-8)

source.connect([emt.SimNode.gnd, load_bus])
load.connect([load_bus])

system = SystemTopology(FREQUENCY, [load_bus], [source, load])
print("Network created: source -> load_bus -> PQLoad")

## Configure and run the simulation

DPsim's Python bindings currently expose switch events only. This notebook therefore advances the simulation one time step at a time and changes the source/load attributes immediately before the scheduled step. This is equivalent to changing the same attributes with C++ `AttributeEvent`s.

In [ ]:
Logger.set_log_dir(str(Path("logs") / SIM_NAME))
logger = Logger(SIM_NAME)
logger.log_attribute("v_load", load.attr("v_intf"))
logger.log_attribute("i_load", load.attr("i_intf"))
logger.log_attribute("p_reference", load.attr("P"))
logger.log_attribute("q_reference", load.attr("Q"))
logger.log_attribute("p_load", load.attr("p_inst"))
logger.log_attribute("q_load", load.attr("q_inst"))
logger.log_attribute("iterations", load.attr("NIterations"))

simulation = Simulation(SIM_NAME)
simulation.set_system(system)
simulation.set_domain(Domain.EMT)
simulation.set_solver(Solver.MNA)
simulation.set_time_step(TIME_STEP)
simulation.set_final_time(FINAL_TIME)
simulation.add_logger(logger)

event_steps = {
    int(round(row.time_s / TIME_STEP)): row for row in EVENTS.itertuples(index=False)
}
number_of_steps = int(round(FINAL_TIME / TIME_STEP))

simulation.start()
try:
    for step in range(1, number_of_steps + 1):
        event = event_steps.get(step)
        if event is not None:
            if event.attribute == "V_ll_rms":
                source.V_ref = three_phase_voltage(event.new_value)
            elif event.attribute == "P":
                load.P = event.new_value
            elif event.attribute == "Q":
                load.Q = event.new_value
            else:
                raise ValueError(f"Unsupported event: {event.attribute}")
        simulation.next()
finally:
    simulation.stop()

csv_path = Path(Logger.get_log_dir()) / f"{SIM_NAME}.csv"
print("Simulation finished. Log:", csv_path.resolve())

## Read and validate the results

The validation is independent of the component's logged power values: P and Q are recalculated from the three instantaneous phase voltages and currents. For a balanced three-phase system, the expected RMS phase current is $|S|/(\sqrt{3} V_{LL})$.

In [ ]:
data = pd.read_csv(csv_path, skipinitialspace=True)
data.columns = data.columns.str.strip()

required_columns = [
    "time",
    *(f"v_load_{phase}" for phase in range(3)),
    *(f"i_load_{phase}" for phase in range(3)),
    "p_reference",
    "q_reference",
    "p_load",
    "q_load",
    "iterations",
]
missing_columns = sorted(set(required_columns) - set(data.columns))
if missing_columns:
    raise KeyError(
        f"Missing columns: {missing_columns}. Available columns: {data.columns.tolist()}"
    )

print(f"Loaded {len(data):,} samples with {len(data.columns)} columns.")
data.head()

In [ ]:
voltage = data[[f"v_load_{phase}" for phase in range(3)]].to_numpy()
current = data[[f"i_load_{phase}" for phase in range(3)]].to_numpy()
voltage_no_zero_sequence = voltage - voltage.mean(axis=1, keepdims=True)
voltage_quadrature = np.column_stack(
    (
        (voltage_no_zero_sequence[:, 1] - voltage_no_zero_sequence[:, 2])
        / np.sqrt(3.0),
        (voltage_no_zero_sequence[:, 2] - voltage_no_zero_sequence[:, 0])
        / np.sqrt(3.0),
        (voltage_no_zero_sequence[:, 0] - voltage_no_zero_sequence[:, 1])
        / np.sqrt(3.0),
    )
)

data["P_calculated"] = np.sum(voltage_no_zero_sequence * current, axis=1)
data["Q_calculated"] = np.sum(voltage_quadrature * current, axis=1)
data["V_ll_rms"] = np.linalg.norm(voltage_no_zero_sequence, axis=1)
data["I_phase_rms"] = np.linalg.norm(current, axis=1) / np.sqrt(3.0)
data["I_expected"] = np.hypot(data.p_reference, data.q_reference) / (
    np.sqrt(3.0) * data.V_ll_rms
)

np.testing.assert_allclose(data.p_load, data.p_reference, rtol=1e-8, atol=5e-2)
np.testing.assert_allclose(data.q_load, data.q_reference, rtol=1e-8, atol=5e-2)
np.testing.assert_allclose(data.P_calculated, data.p_reference, rtol=1e-8, atol=5e-2)
np.testing.assert_allclose(data.Q_calculated, data.q_reference, rtol=1e-8, atol=5e-2)
np.testing.assert_allclose(data.I_phase_rms, data.I_expected, rtol=1e-7, atol=1e-5)

print(f"max |P - P_ref| = {np.max(np.abs(data.P_calculated - data.p_reference)):.3e} W")
print(
    f"max |Q - Q_ref| = {np.max(np.abs(data.Q_calculated - data.q_reference)):.3e} var"
)
print(f"max current error = {np.max(np.abs(data.I_phase_rms - data.I_expected)):.3e} A")
print(f"maximum corrector iterations = {int(data.iterations.max())}")
print("All numerical checks passed.")

In [ ]:
operating_windows = [
    ("initial", 0.04, 0.09),
    ("low voltage", 0.14, 0.19),
    ("high voltage", 0.24, 0.29),
    ("P step", 0.34, 0.39),
    ("Q step", 0.44, 0.49),
]
summary_rows = []
for case, start, end in operating_windows:
    interval = data[data.time.between(start, end)]
    summary_rows.append(
        {
            "case": case,
            "V_ll_rms [V]": interval.V_ll_rms.mean(),
            "I_phase_rms [A]": interval.I_phase_rms.mean(),
            "I_expected [A]": interval.I_expected.mean(),
            "P [kW]": interval.P_calculated.mean() / 1e3,
            "Q [kvar]": interval.Q_calculated.mean() / 1e3,
        }
    )
summary = pd.DataFrame(summary_rows).set_index("case")
summary.round(3)

## Visualization

The dashed curves are references or analytical expectations; the solid curves are simulated or independently reconstructed values.

In [ ]:
voltage_reference = np.select(
    [data.time < 0.10, data.time < 0.20], [400.0, 320.0], default=440.0
)
figure, axes = plt.subplots(4, 1, figsize=(11, 10), sharex=True)

axes[0].plot(data.time, data.V_ll_rms, label="simulated")
axes[0].plot(data.time, voltage_reference, "--", label="reference")
axes[0].set_ylabel(r"$V_{LL,RMS}$ [V]")
axes[1].plot(data.time, data.I_phase_rms, label="simulated")
axes[1].plot(data.time, data.I_expected, "--", label=r"$|S|/(\sqrt{3}V_{LL})$")
axes[1].set_ylabel(r"$I_{phase,RMS}$ [A]")
axes[2].plot(data.time, data.P_calculated / 1e3, label="calculated")
axes[2].plot(data.time, data.p_reference / 1e3, "--", label="reference")
axes[2].set_ylabel("P [kW]")
axes[3].plot(data.time, data.Q_calculated / 1e3, label="calculated")
axes[3].plot(data.time, data.q_reference / 1e3, "--", label="reference")
axes[3].set_ylabel("Q [kvar]")
axes[3].set_xlabel("time [s]")

for axis in axes:
    for event_time in EVENTS.time_s:
        axis.axvline(event_time, color="0.5", linewidth=0.8, alpha=0.5)
    axis.grid(True, alpha=0.3)
    axis.legend(loc="best")

figure.suptitle("EMT three-phase PQLoad response")
figure.tight_layout()

In [ ]:
detail = data[data.time.between(0.28, 0.32)]
figure, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
for phase, phase_name in enumerate("abc"):
    axes[0].plot(detail.time, detail[f"v_load_{phase}"], label=f"v_{phase_name}")
    axes[1].plot(detail.time, detail[f"i_load_{phase}"], label=f"i_{phase_name}")

for axis in axes:
    axis.axvline(0.30, color="black", linestyle="--", linewidth=1.0, label="P step")
    axis.grid(True, alpha=0.3)
    axis.legend(ncol=4)

axes[0].set_ylabel("phase voltage [V]")
axes[1].set_ylabel("phase current [A]")
axes[1].set_xlabel("time [s]")
figure.suptitle("Instantaneous abc waveforms around the active-power step")
figure.tight_layout()

## Interpretation

The numerical assertions and plots test three defining properties of the model:

1. P and Q follow their set points.
2. At unchanged P and Q, current rises when voltage falls and falls when voltage rises.
3. Increasing P or Q increases the current magnitude while the source holds the terminal voltage.

This is the expected behavior of a constant-PQ load above its configured minimum-voltage limit.